In [ ]:
# Install LangChain (the LLM framework) and langchain-openai (the GPT model integration).
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain.agents import AgentState
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
# ^ Message types: HumanMessage = user input, AIMessage = model reply,
#   SystemMessage = instructions to the model, BaseMessage = common base type
from langchain_core.runnables import Runnable
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode
from pathlib import Path
from pydantic import SecretStr
from typing import List

# Load the OpenAI API key from Colab secrets — never hardcode API keys directly in a notebook
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper: pretty-prints every message in a conversation (user, AI, tool calls, tool results)
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper: renders the compiled graph as a PNG image and shows it inline in Jupyter
def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
# --- Tool definitions ---
# Tools are Python functions the LLM can choose to call when answering a question.
# The @tool decorator registers them as LangChain tools.
# IMPORTANT: The docstring tells the LLM WHAT the tool does and WHEN to use it —
#            the model reads the docstring to decide which tool to call.

@tool
def weather(city: str) -> str:
    """Return a (fake) current-weather report for a city."""
    # Simulated weather data — in a real app this would call an external weather API
    data = {
        "sofia": "Sofia: 18 C, partly cloudy",
        "london": "London: 11 C, rainy",
        "tokyo": "Tokyo: 22 C, sunny",
    }
    # .get() returns a fallback message if the city isn't in the dataset
    return data.get(city.lower(), f"No data for {city}.")

@tool
def search(query: str) -> str:
    """Look up a term in the built-in mini-encyclopedia."""
    # Simulated knowledge base — in a real app this would call a search API or vector DB
    data = {
        "langgraph": "LangGraph is a library for building stateful, cyclic LLM apps.",
        "react": "ReAct is a prompting pattern: Reason then Act, in a loop.",
        "dag": "A DAG is a directed acyclic graph — no cycles allowed.",
    }
    return data.get(query.lower(), "(nothing found)")


# The system prompt is prepended to every model call — it sets the AI's behaviour
SYSTEM_PROMPT = "You are a concise assistant. Use tools when useful."

# The list of tools we give the model access to
TOOLS = [weather, search]

In [ ]:
# Initialize the GPT model and bind our tools to it.
# .bind_tools() informs the model about the available tools so it can decide when to call them.
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(TOOLS)

def model_node(state: AgentState):
    # Prepend the system prompt to the full conversation history, then invoke the model.
    # state["messages"] contains everything said so far (human messages, tool results, etc.)
    response = model.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    # Return only the new AI response — LangGraph appends it to the messages list in the state
    return { "messages": [response] }

In [ ]:
# Routing (conditional edge) function — checks if the model wants to call a tool.
# Returns True  => the model's last message contains tool calls -> go to the "tools" node
# Returns False => the model produced a final text answer   -> end the graph
def has_pending_tool_calls(state: AgentState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False  # Nothing to check yet

    last_message = messages[-1]
    # AIMessage.tool_calls is a non-empty list when the model requested one or more tool calls
    return isinstance(last_message, AIMessage) and last_message.tool_calls

In [ ]:
# Assemble the ReAct agent loop using LangGraph.
# ReAct pattern: model reasons about whether to use a tool, acts (calls it), observes the result,
# then reasons again — looping until it has a final answer.
#
# Graph flow:
#   START -> model -> (tool call?) -> tools -> model -> (repeat) -> END
graph_builder = StateGraph(AgentState)

# Register the model node and the built-in ToolNode (handles executing the actual tool calls)
graph_builder.add_node("model", model_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

# Always start by calling the model
graph_builder.add_edge(START, "model")
graph_builder.add_conditional_edges("model", lambda x: "tools" if has_pending_tool_calls(x) else END, ["tools", END])
graph_builder.add_edge("tools", "model")

# Compile into a runnable agent
graph = graph_builder.compile()

In [ ]:
# Visualize the agent loop — you should see the model <-> tools cycle clearly
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Run the agent with an initial user question.
# The agent will autonomously decide whether to call tools and keep looping
# until it produces a final answer with no more tool calls.
final_state = graph.invoke(
    input={
        "messages": [HumanMessage("What's the weather in Tokyo and what is LangGraph, briefly?")]
    }
)

In [ ]:
# Print the full conversation: user question -> tool calls -> tool results -> final AI answer
print_conversation(final_state["messages"])

In [ ]:
# Inspect the raw final state dictionary — useful to see the raw message objects in detail
final_state